# 05 · Model comparison
Compares Logistic Regression, Random Forest and XGBoost using 5-fold stratified cross-validation (CV) on the training set only. **Selection rule, fixed before running:** the model with the highest mean cross-validated PR-AUC is selected and saved for notebook 06.

## 1. Setup: load the training data and define the folds
**What it does:** imports the libraries, loads `train.csv` and creates the same 5 stratified folds (seed 42) used in earlier notebooks.

**Why:** every model is scored on identical folds, so the comparison is fair. The test set is not loaded.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
pd.set_option("display.width", 200)

train = pd.read_csv("data/train.csv")
X_train = train.drop(columns="Bankrupt?")
y_train = train["Bankrupt?"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. Define the three candidate models
**What it does:** builds Logistic Regression (inside a pipeline with `log1p` and standardisation), Random Forest and XGBoost, all with fixed random seeds and no class weights.

**Why:** tree models split on value order, so they use the raw ratios; class weights were dropped after notebook 04 showed they did not improve PR-AUC. Default settings are used (no tuning).

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("log", FunctionTransformer(np.log1p)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", n_jobs=-1, random_state=42),
}

## 3. Cross-validated ranking quality
**What it does:** runs 5-fold CV for each model and reports the mean and standard deviation of PR-AUC and ROC-AUC.

**Why:** these metrics measure how well each model ranks bankrupt records above healthy ones. The standard deviation shows how much results move between folds, so small gaps are not over-read.

In [ ]:
rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
    rows.append({"model": name,
                 "PR-AUC mean": s["test_average_precision"].mean(),
                 "PR-AUC std": s["test_average_precision"].std(),
                 "ROC-AUC mean": s["test_roc_auc"].mean(),
                 "ROC-AUC std": s["test_roc_auc"].std()})
cv_results = pd.DataFrame(rows).set_index("model").round(3)
print(cv_results)

                     PR-AUC mean  PR-AUC std  ROC-AUC mean  ROC-AUC std
model                                                                  
Logistic Regression        0.380       0.043         0.912        0.022
Random Forest              0.442       0.051         0.938        0.024
XGBoost                    0.459       0.061         0.940        0.020


## 4. Bankruptcies caught at equal review workload
**What it does:** produces out-of-fold risk scores for every training record, then counts how many bankruptcies each model finds when analysts review the top 5%, 10%, 15% and 20% of records.

**Why:** translates the metrics into the business question: with the same analyst workload, which model finds more bankruptcies?

In [ ]:
def capacity_table(scores, y_true, capacities=(0.05, 0.10, 0.15, 0.20)):
    y_true = np.asarray(y_true)
    order = np.argsort(-scores)
    out = []
    for cap in capacities:
        k = int(round(cap * len(scores)))
        caught = int(y_true[order[:k]].sum())
        out.append({"capacity": f"{int(cap*100)}%", "companies_reviewed": k,
                    "bankruptcies_caught": caught,
                    "recall": round(caught / y_true.sum(), 3),
                    "precision": round(caught / k, 3),
                    "score_threshold": float(scores[order[k - 1]])})
    return pd.DataFrame(out)

risk_scores = {name: cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
               for name, m in models.items()}
caught = pd.DataFrame({name: capacity_table(sc, y_train).set_index("capacity")["bankruptcies_caught"]
                       for name, sc in risk_scores.items()})
caught.insert(0, "companies_reviewed", capacity_table(risk_scores["XGBoost"], y_train)
              .set_index("capacity")["companies_reviewed"])
print(caught)

          companies_reviewed  Logistic Regression  Random Forest  XGBoost
capacity                                                                 
5%                       273                   99            104      104
10%                      546                  124            134      139
15%                      818                  138            154      149
20%                     1091                  156            160      160


## 5. Apply the selection rule and save the winner
**What it does:** picks the model with the highest mean CV PR-AUC and writes its name to `results/selected_model.txt`, with the full comparison in `results/cv_results.csv`.

**Why:** notebook 06 reads this file, so the final evaluation always uses the model chosen here.

In [ ]:
import os
os.makedirs("results", exist_ok=True)
cv_results.to_csv("results/cv_results.csv")
selected = cv_results["PR-AUC mean"].idxmax()
with open("results/selected_model.txt", "w") as f:
    f.write(selected)
print("Selected model (highest mean CV PR-AUC):", selected)

Selected model (highest mean CV PR-AUC): XGBoost
